In [7]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [8]:
import torch
from torchvision import models
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)

model.load_state_dict(torch.load("model_01_simple_cnn_best.pth", map_location=device))
model.eval()

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [10]:
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [11]:
from PIL import Image
import torch

def predict_image(model, image_path, transform, device):
    model.eval()

    img = Image.open(image_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img)
        probs = torch.softmax(outputs, dim=1)
        pred = torch.argmax(probs, dim=1)

    class_names = ["NORMAL", "PNEUMONIA"]

    predicted_class = class_names[pred.item()]
    confidence = probs.max().item()

    print(f"Prediction: {predicted_class}")
    print(f"Confidence: {confidence:.4f}")

    return predicted_class, confidence

In [16]:
img = Image.open(r"C:\Users\Nourhan Yehia\Desktop\Jupyter\xray_pneumonia_classifier\data\processed\test\PNEUMONIA\person1_virus_7.jpeg")
img = transform(img).unsqueeze(0).to(device)

In [17]:
with torch.no_grad():
    outputs = model(img)
    probs = torch.softmax(outputs, dim=1)
    pred = torch.argmax(probs, dim=1)

classes = ["NORMAL", "PNEUMONIA"]

predicted_class = classes[pred.item()]
confidence = probs.max().item()

print(f"Prediction: {predicted_class}")
print(f"Confidence: {confidence:.4f}")


Prediction: PNEUMONIA
Confidence: 1.0000
